<a href="https://colab.research.google.com/github/arunimamuralitharan/aphasia_qwen_FYP/blob/main/demo1_qwen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q transformers accelerate peft librosa pandas

In [3]:
from google.colab import drive
import os
drive.mount('/content/drive')
zip_path = '/content/drive/MyDrive/final_thesis_dataset_demo1.zip'
local_dir = '/content/processed_dataset'
if not os.path.exists(local_dir):
    os.makedirs(local_dir)
    print("Unzipping dataset to local disk")
    !unzip -q {zip_path} -d {local_dir}
    print("Unzip complete")
else:
    print("Dataset already unzipped.")
file_count = len(os.listdir(local_dir))
print(f"Total files in local directory: {file_count}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset already unzipped.
Total files in local directory: 815


In [4]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/metadata.csv')

In [8]:
import os
!ls -d /content/*/ | grep dataset
!ls /content/drive/MyDrive/final_thesis_dataset_demo1.zip | head -n 5

/content/processed_dataset/
/content/drive/MyDrive/final_thesis_dataset_demo1.zip


In [5]:
import os
unzip_root = "/content/processed_dataset"
subfolders = [f for f in os.listdir(unzip_root) if os.path.isdir(os.path.join(unzip_root, f))]

if subfolders:
    AUDIO_DIR = os.path.join(unzip_root, subfolders[0])
    print(f"Found subfolder. Updating AUDIO_DIR to: {AUDIO_DIR}")
else:
    AUDIO_DIR = unzip_root
    print(f"No subfolders. Files are in: {AUDIO_DIR}")

test_files = os.listdir(AUDIO_DIR)
print(f"First 3 files: {test_files[:3]}")

No subfolders. Files are in: /content/processed_dataset
First 3 files: ['healthy_SBC0017_5.wav', 'aphasia_1713_9.wav', 'aphasia_1944_55.wav']


In [6]:
import os
import torch
import pandas as pd
import librosa
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor

# CONFIGURATION
AUDIO_DIR = "/content/processed_dataset"
CSV_PATH = "/content/drive/MyDrive/metadata.csv"
MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"

# DATASET CLASS
class AphasiaDataset(Dataset):
    def __init__(self, csv_file, processor, audio_dir):
        self.data = pd.read_csv(csv_file)
        self.processor = processor
        self.audio_dir = audio_dir
        self.sampling_rate = self.processor.feature_extractor.sampling_rate

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        raw_file_path = self.data.iloc[idx]['file_name']
        file_name = os.path.basename(raw_file_path)
        audio_path = os.path.join(self.audio_dir, file_name)

        label_id = self.data.iloc[idx]['label']
        label_text = "Aphasia" if label_id == 1 else "Control"

        # 1.Load audio correctly
        audio, _ = librosa.load(audio_path, sr=self.sampling_rate)

        # 2.Extract audio features explicitly
        audio_inputs = self.processor.feature_extractor(
            audio,
            sampling_rate=self.sampling_rate,
            return_tensors="pt"
        )

        # 3.Process text with template
        conversation = [
            {"role": "user", "content": [
                {"type": "audio", "audio_url": audio_path},
                {"type": "text", "text": "Does this speech sample show signs of aphasia?"}
            ]},
            {"role": "assistant", "content": [{"type": "text", "text": label_text}]}
        ]

        text = self.processor.apply_chat_template(conversation, add_generation_prompt=False, tokenize=False)
        text_inputs = self.processor.tokenizer(text, return_tensors="pt", padding=True)

        # 4.Return the exact keys Qwen2-Audio expects
        return {
            "input_ids": text_inputs["input_ids"].squeeze(0),
            "audio_values": audio_inputs["input_features"].squeeze(0),
            "labels": text_inputs["input_ids"].squeeze(0).clone()
        }

#DATA COLLATOR
def collate_fn(batch):
    input_ids = [item["input_ids"] for item in batch]
    audio_values = [item["audio_values"] for item in batch]
    labels = [item["labels"] for item in batch]

    input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=processor.tokenizer.pad_token_id)
    labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100)
    audio_values = torch.stack(audio_values)

    return {
        "input_ids": input_ids,
        "audio_values": audio_values,
        "labels": labels
    }

# CPU TEST
print("CPU Check...") #Pre-flighting
try:
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    test_dataset = AphasiaDataset(CSV_PATH, processor, AUDIO_DIR)

    test_loader = DataLoader(test_dataset, batch_size=2, collate_fn=collate_fn)
    first_batch = next(iter(test_loader))

    print(f"Samples: {len(test_dataset)}")
    print(f"Batch Audio Shape: {first_batch['audio_values'].shape}")
    print("\nReady for the A100 GPU.")
except Exception as e:
    print(f"\nERROR: {e}")

CPU Check...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Samples: 815
Batch Audio Shape: torch.Size([2, 128, 3000])

Ready for the A100 GPU.


In [1]:
!pip install -U -q bitsandbytes>=0.46.1 transformers accelerate peft trl librosa

In [11]:
from transformers import Qwen2AudioForConditionalGeneration, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

# 1. 4-bit Quantization Config (NF4)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 2. Load the Model with SDPA (Native PyTorch Attention)
print("Loading Qwen2-Audio-7B...")
model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa" \
)

# 3. Training Prep
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# 4. LoRA Setup
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
print("Model loaded successfully on L4 using SDPA.")

Loading Qwen2-Audio-7B...


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/876 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

Model loaded successfully on L4 using SDPA.


In [15]:
import torch

def run_baseline(sample_idx):
    # 1. Get a sample
    sample = test_dataset[sample_idx]

    # 2. Prepare inputs with strictly enforced dtypes
    model_inputs = {
        "input_ids": sample["input_ids"].unsqueeze(0).to("cuda"),
        "input_features": sample["audio_values"].unsqueeze(0).to("cuda").to(torch.bfloat16),
        # FIX: Explicitly cast masks to Long to avoid Float bitwise_or error
        "attention_mask": torch.ones(1, sample["input_ids"].shape[0], dtype=torch.long).to("cuda"),
        "feature_attention_mask": torch.ones(1, sample["audio_values"].shape[0], dtype=torch.long).to("cuda")
    }

    # 3. Generate
    print(f"\n--- Testing Sample {sample_idx} ---")
    with torch.no_grad():
        output_ids = model.generate(
            **model_inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=processor.tokenizer.pad_token_id
        )

    # 4. Decode
    input_len = model_inputs["input_ids"].shape[1]
    response_ids = output_ids[0][input_len:]
    response = processor.batch_decode(response_ids, skip_special_tokens=True)[0]

    print(f"Model Baseline Output: {response}")

# Run tests
run_baseline(0)
run_baseline(1)


--- Testing Sample 0 ---
Model Baseline Output: 
going

to take a little
 the sentence contain have any emotion


--- Testing Sample 1 ---
Model Baseline Output: 






image contain a fearful
?


In [24]:
from transformers import Trainer, TrainingArguments
import torch

model.config.use_cache = False
if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()

# Training Args
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/aphasia_qwen_checkpoints",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-5,
    num_train_epochs=5,
    bf16=True,
    logging_steps=5,
    save_strategy="epoch",
    eval_strategy="no",
    optim="paged_adamw_8bit",
    gradient_checkpointing=False,
    remove_unused_columns=False,
    report_to="none"
)

# Trainer Init
trainer = Trainer(
    model=model,
    train_dataset=test_dataset,
    args=training_args,
    data_collator=fine_tuning_collator
)

# Start
print(" Training...")
trainer.train()

trainer.save_model("/content/drive/MyDrive/aphasia_qwen_final_adapter")

 Training...


Step,Training Loss
5,53.994946
10,36.449991
15,26.349887
20,19.402075
25,15.135658
30,11.578793
35,9.402947
40,7.788622
45,7.212042
50,7.026346


In [25]:
trainer.save_model("/content/drive/MyDrive/aphasia_qwen_manual_backup")
print("Manual backup saved!")

Manual backup saved!


In [28]:
def sanity_check(num=3):
    model.eval()
    for i in range(num):
        sample = test_dataset[i]
        inputs = {
            "input_ids": sample["input_ids"].unsqueeze(0).to("cuda"),
            "input_features": sample["audio_values"].unsqueeze(0).to("cuda").to(torch.bfloat16),
            "attention_mask": torch.ones(1, sample["input_ids"].shape[0], dtype=torch.long).to("cuda"),
            "feature_attention_mask": torch.ones(1, sample["audio_values"].shape[0], dtype=torch.long).to("cuda")
        }
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=128)
            full_text = processor.decode(out[0], skip_special_tokens=True)

        print(f"\n--- Sample {i} RAW OUTPUT ---")
        print(full_text)
        print(f"--- GROUND TRUTH ---")
        print(processor.decode(sample["labels"], skip_special_tokens=True))

sanity_check(3)


--- Sample 0 RAW OUTPUT ---
system
You are a helpful assistant.
user
Audio 1: 
Does this speech sample show signs of aphasia?
assistant
Aphasia

--- GROUND TRUTH ---
system
You are a helpful assistant.
user
Audio 1: 
Does this speech sample show signs of aphasia?
assistant
Aphasia


--- Sample 1 RAW OUTPUT ---
system
You are a helpful assistant.
user
Audio 1: 
Does this speech sample show signs of aphasia?
assistant
Aphasia

--- GROUND TRUTH ---
system
You are a helpful assistant.
user
Audio 1: 
Does this speech sample show signs of aphasia?
assistant
Aphasia


--- Sample 2 RAW OUTPUT ---
system
You are a helpful assistant.
user
Audio 1: 
Does this speech sample show signs of aphasia?
assistant
Aphasia

--- GROUND TRUTH ---
system
You are a helpful assistant.
user
Audio 1: 
Does this speech sample show signs of aphasia?
assistant
Aphasia



In [31]:
import random
from sklearn.metrics import classification_report, confusion_matrix
import torch
import numpy as np

def run_final_metrics_v3(num_eval_samples=60):
    model.eval()
    y_true = []
    y_pred = []

    # 1. Shuffle indices to get a mix of Aphasia (1) and Control (0)
    all_indices = list(range(len(test_dataset)))
    # Set seed for reproducibility
    random.seed(42)
    eval_indices = random.sample(all_indices, num_eval_samples)

    print(f"Evaluating {num_eval_samples} randomly sampled files...")

    for count, idx in enumerate(eval_indices):
        sample = test_dataset[idx]

        # Ground Truth
        gt_text = processor.decode(sample["labels"], skip_special_tokens=True)
        true_class = 1 if "Aphasia" in gt_text else 0

        # Inference
        model_inputs = {
            "input_ids": sample["input_ids"].unsqueeze(0).to("cuda"),
            "input_features": sample["audio_values"].unsqueeze(0).to("cuda").to(torch.bfloat16),
            "attention_mask": torch.ones(1, sample["input_ids"].shape[0], dtype=torch.long).to("cuda"),
            "feature_attention_mask": torch.ones(1, sample["audio_values"].shape[0], dtype=torch.long).to("cuda")
        }

        with torch.no_grad():
            output_ids = model.generate(**model_inputs, max_new_tokens=64)
            full_response = processor.decode(output_ids[0], skip_special_tokens=True)

        # Parse prediction
        if "assistant" in full_response:
            generated_answer = full_response.split("assistant")[-1]
            pred_class = 1 if "Aphasia" in generated_answer else 0
        else:
            pred_class = 0

        y_true.append(true_class)
        y_pred.append(pred_class)

        if count % 10 == 0:
            print(f"Processed {count}/{num_eval_samples}...")

    # 2. Print the Report
    print("\n" + "="*40)
    print("      SHUFFLED CLINICAL PERFORMANCE")
    print("="*40)
    print(classification_report(y_true, y_pred, labels=[0, 1], target_names=["Control", "Aphasia"], zero_division=0))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred, labels=[0, 1]))

run_final_metrics_v3(60)

Evaluating 60 randomly sampled files...
Processed 0/60...
Processed 10/60...
Processed 20/60...
Processed 30/60...
Processed 40/60...
Processed 50/60...

      SHUFFLED CLINICAL PERFORMANCE
              precision    recall  f1-score   support

     Control       1.00      1.00      1.00        13
     Aphasia       1.00      1.00      1.00        47

    accuracy                           1.00        60
   macro avg       1.00      1.00      1.00        60
weighted avg       1.00      1.00      1.00        60


Confusion Matrix:
[[13  0]
 [ 0 47]]
